[![Kaggle](https://img.shields.io/badge/Notebook-Kaggle-yellow.svg)](https://www.kaggle.com/code/radimkzl/fine-tuning-qwen-2-5-text2cypher)

## **Installing Unsloth and dependencies**
### 1. Optimized Unsloth installation for the Kaggle environment

In [1]:
import os

os.system('pip install --upgrade "huggingface-hub>=1.31.0,<2.0"')
os.system("pip install --no-deps unsloth unsloth_zoo")
os.system('pip install --no-deps peft accelerate bitsandbytes xformers')
os.system('pip install --no-deps --upgrade trl')
os.system("pip install datasets evaluate")

print("\n" + "=" * 60)
print("🚀 Installation complete. Forcing a restart of the Python kernel...")
print("=" * 60 + "\n")

#os._exit(0)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 5.0 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.29.0
    Uninstalling huggingface_hub-1.29.0:
      Successfully uninstalled huggingface_hub-1.29.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.8/79.8 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.2/30.2 MB 72.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 85.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 91.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 996.6/996.6 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.3 MB/s eta 0:00:00

🚀 Installation complete. Forcing a restart of the Python kernel...



In [2]:
import importlib.metadata as m
for p in ["huggingface_hub", "transformers", "datasets", "tokenizers", "unsloth", "unsloth_zoo", "trl", "peft"]:
    try:
        print(f"{p:16} {m.version(p)}")
    except m.PackageNotFoundError:
        print(f"{p:16} NOT INSTALLED")

huggingface_hub  1.33.0
transformers     5.16.1
datasets         4.8.5
tokenizers       0.23.1
unsloth          2026.10.11
unsloth_zoo      2026.10.11
trl              1.17.1
peft             0.20.0


## **Authentication, Git configuration, and HF login**

In [3]:
import subprocess
from huggingface_hub import login, HfApi, ModelCard, ModelCardData

import importlib.metadata as m
print(m.version("trl"))

# Safe import for Google Colab
try:
    from google.colab import userdata
except ImportError:
    userdata = None

# Safe Import for Kaggle Secrets
try:
    from kaggle_secrets import UserSecretsClient
    # Handling for cases where UserSecretsError does not exist in the given version
    try:
        from kaggle_secrets import UserSecretsError
    except ImportError:
        UserSecretsError = Exception
except ImportError:
    UserSecretsClient = None
    UserSecretsError = Exception

import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments
from unsloth.chat_templates import train_on_responses_only

1.17.1
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


##  **Loading secrets from Kaggle (with fallback to Colab/env)**

In [4]:
try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    user_email = user_secrets.get_secret("USER_EMAIL")
    user_name = user_secrets.get_secret("USER_NAME")
    hf_username = user_secrets.get_secret("HF_USERNAME")
except Exception:
    try:
        hf_token = userdata.get("HF_TOKEN")
        user_email = userdata.get("USER_EMAIL")
        user_name = userdata.get("USER_NAME")
        hf_username = userdata.get("HF_USERNAME")
    except Exception as e:
        print(f"Warning: Failed to load secrets automatically: {e}")

## **Git configuration settings**

In [5]:
def set_git_config(email, name):
    try:
        subprocess.run(
            ["git", "config", "--global", "user.email", email], check=True
        )
        subprocess.run(["git", "config", "--global", "user.name", name], check=True)
        print(f"Git configuration set for: {name} <{email}>")
    except subprocess.CalledProcessError as e:
        print(f"Error setting Git configuration: {e}")

In [6]:
if user_email and user_name:
    set_git_config(user_email, user_name)

Git configuration set for: Radim Közl <kozl@svmetal.cz>


## **Logging in to Hugging Face Hub**

In [7]:
if hf_token:
    login(token=hf_token)
    print("Successfully logged into Hugging Face Hub!")
else:
    raise ValueError("HF_TOKEN was not found in Kaggle Secrets!")

Successfully logged into Hugging Face Hub!


## **Loading Qwen2.5-Coder-3B & LoRA**

In [8]:
hf_username = 'KRadim'
BASE_MODEL_NAME = "unsloth/Qwen2.5-Coder-3B-Instruct"
HUB_REPO_LORA = f"{hf_username}/Qwen2.5-Coder-3B-text2cypher-LoRA"
HUB_REPO_FULL = f"{hf_username}/Qwen2.5-Coder-3B-text2cypher-Merged"
HUB_REPO_GGUF = f"{hf_username}/Qwen2.5-Coder-3B-text2cypher-GGUF"

In [9]:
max_seq_length = 2048
dtype = None  # Auto-detection (bfloat16 pro T4/P100 fallback na float16)
load_in_4bit = True

In [10]:
api = HfApi()
repo_exists = False

#### *Check if a previous checkpoint already exists on the HF Hub*

In [11]:
repo_exists = False
try:
    files = api.list_repo_files(repo_id=HUB_REPO_LORA, token=hf_token)
    if "adapter_config.json" in files:
        repo_exists = True
        print(f"--> Adapter found: {HUB_REPO_LORA}. Resuming training...")
    else:
        print(f"--> Repo exists but contains no adapter. Starting from base model {BASE_MODEL_NAME}...")
except Exception:
    print(f"--> Repo not found. Starting from base model {BASE_MODEL_NAME}...")

--> Adapter found: KRadim/Qwen2.5-Coder-3B-text2cypher-LoRA. Resuming training...


### **RUN INDEX & SEED SETUP**

In [12]:
RUN_INDEX = 5  # ⚙️ Change manually: 1, 2, 3... (or load automatically)

BASE_SEED = 3407
CURRENT_TRAIN_SEED = BASE_SEED + (RUN_INDEX - 1)

print(f"🚀 RUN INDEX: {RUN_INDEX} | Using Random Seed: {CURRENT_TRAIN_SEED}")

🚀 RUN INDEX: 5 | Using Random Seed: 3411


#### *Loading either an adapter in progress or a clean base model.*

In [13]:
if repo_exists:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=HUB_REPO_LORA,
        max_seq_length=max_seq_length,
        dtype=dtype,
        load_in_4bit=load_in_4bit,
    )
else:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL_NAME,
        max_seq_length=max_seq_length,
        dtype=dtype,
        load_in_4bit=load_in_4bit,
    )
    # Initialize new QLoRA adapter only on the first run
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=[
            "q_proj", 
            "k_proj", 
            "v_proj", 
            "o_proj",
            "gate_proj", 
            "up_proj", 
            "down_proj",
        ],
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=CURRENT_TRAIN_SEED,
    )

==((====))==  Unsloth 2026.10.11: Fast Qwen2 patching. Transformers: 5.16.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Not splitting across GPUs; the model fits on cuda:0 with room to train. Pass device_map = "unsloth_balanced" to split it anyway.


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Unsloth 2026.10.11 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).


### **LOADING AND PREPARING THE NEO4J DATASET (TEXT-TO-CYPHER)**

#### *Loading the official Neo4j dataset from Hugging Face*

In [14]:
DATASET_NAME = "KRadim/multilingual-text2cypher-2025"
print(f"Loading dataset: {DATASET_NAME}...")
dataset_raw = load_dataset(DATASET_NAME)

Loading dataset: KRadim/multilingual-text2cypher-2025...


README.md:   0%|          | 0.00/1.65k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 35.7MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 4.04MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/218074 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/24230 [00:00<?, ? examples/s]

#### *If the dataset has a split (train/test), we take 'train'*

In [15]:
if "train" in dataset_raw:
  train_dataset = dataset_raw["train"]
else:
  train_dataset = dataset_raw

#### *Cropping the dataset for the current RUN_INDEX*

In [16]:
SUBSAMPLE_SIZE = 35000
start_idx = (RUN_INDEX - 1) * SUBSAMPLE_SIZE
end_idx = start_idx + SUBSAMPLE_SIZE

In [17]:
train_dataset = train_dataset.select(range(start_idx, min(end_idx, len(train_dataset))))
print(f"🔄 RUN #{RUN_INDEX}: Training on the dataset from index {start_idx} to {end_idx} ({len(train_dataset)} samples).")

🔄 RUN #5: Training on the dataset from index 140000 to 175000 (35000 samples).


#### *Function to convert dataset structure to Qwen ChatML format*

In [18]:
def format_cypher_prompts(examples):
    texts = []
    
    questions = examples["question"]
    schemas = examples["schema"]
    cyphers = examples["cypher"]

    for q, s, c in zip(questions, schemas, cyphers):
        system_content = (
            "You are an expert in Neo4j databases and the Cypher query language. "
            "Your task is to convert natural language queries into accurate and efficient Cypher queries "
            "based on the provided database schema.\n\n"
            f"### Database Schema:\n{s}"
        )

        messages = [
            {"role": "system", "content": system_content},
            {"role": "user", "content": q},
            {"role": "assistant", "content": f"```cypher\n{c}\n```"},
        ]

        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        texts.append(text)

    return {"text": texts}

#### *Applying formatting to the entire dataset*

In [19]:
print("I am formatting the data into the ChatML structure...")
dataset = train_dataset.map(format_cypher_prompts, batched=True)

I am formatting the data into the ChatML structure...


Map:   0%|          | 0/35000 [00:00<?, ? examples/s]

In [20]:
print(f"Dataset prepared! Total training examples: {len(dataset)}")

Dataset prepared! Total training examples: 35000


### **Training configuration (SFTTrainer with epoch definition)**

#### *Target training parameters*

In [21]:
TARGET_EPOCHS = 1  # Set the total number of epochs required

In [22]:
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_length=max_seq_length,
        dataset_num_proc=2,
        packing=False,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=TARGET_EPOCHS,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=CURRENT_TRAIN_SEED,
        data_seed=CURRENT_TRAIN_SEED,
        output_dir="outputs",
        save_strategy="epoch",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/35000 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


#### *Input masking (training only on the response – Cypher code)*

In [23]:
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

Map (num_proc=2):   0%|          | 0/35000 [00:00<?, ? examples/s]

Filter (num_proc=2):   0%|          | 0/35000 [00:00<?, ? examples/s]

Unsloth: Removed 3470 out of 35000 samples from train_dataset where all labels were -100 (no response marker found, usually truncation). This prevents NaN loss during training.


### *Starting the training session, with the option to resume*

In [24]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 31,530 | Num Epochs = 1 | Total steps = 3,942
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,0.050811
2,0.056785
3,0.077798
4,0.077745
5,0.041466
6,0.074544
7,0.109579
8,0.021631
9,0.062475
10,0.039814


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-3942/tokenizer_config.json.


### **Conditional saving (LoRA vs. Merged 16-bit & GGUF)**

In [25]:
current_epoch = trainer.state.epoch
print(f"Epochs currently completed: {current_epoch} / {TARGET_EPOCHS}")

Epochs currently completed: 1.0 / 1


#### *Always save/update intermediate LoRA adapters*

In [26]:
print(f"Saving intermediate LoRA adapters to: {HUB_REPO_LORA}")
model.push_to_hub(HUB_REPO_LORA, token=hf_token)
tokenizer.push_to_hub(HUB_REPO_LORA, token=hf_token)

Saving intermediate LoRA adapters to: KRadim/Qwen2.5-Coder-3B-text2cypher-LoRA


README.md:   0%|          | 0.00/1.45k [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

Saved model to https://huggingface.co/KRadim/Qwen2.5-Coder-3B-text2cypher-LoRA


Unsloth: Restored added_tokens_decoder metadata in /tmp/tmpdf7898_h/tokenizer_config.json.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.


In [27]:
if current_epoch >= TARGET_EPOCHS:
    print("\n" + "=" * 50)
    print("Target epochs reached! Exporting 16-bit merged and GGUF models...")
    print("=" * 50)

    # 1. Push 16-bit Merged Model
    print(f"1. Uploading merged 16-bit model to: {HUB_REPO_FULL}")
    model.push_to_hub_merged(
        HUB_REPO_FULL,
        tokenizer,
        save_method="merged_16bit",
        token=hf_token,
    )

    # 2. Push GGUF Model (Q4_K_M)
    print(f"2. Converting and uploading GGUF (Q4_K_M) to: {HUB_REPO_GGUF}")
    model.push_to_hub_gguf(
        HUB_REPO_GGUF,
        tokenizer,
        quantization_method="q4_k_m",
        token=hf_token,
    )

    # 3. Create and Upload Model Card
    print("3. Generating and uploading Hugging Face Model Card...")

    # Safely obtaining the number of examples
    dataset_size = len(dataset) if "dataset" in locals() else len(train_dataset)
    
    card_data = ModelCardData(
        language="en",
        license="apache-2.0",
        library_name="peft",
        tags=["text-to-cypher", "neo4j", "cypher", "unsloth", "qwen2.5"],
        datasets=[DATASET_NAME],
        pipeline_tag="text-generation",
    )

    model_card_text = f"""
# Qwen2.5-Coder-3B Text-to-Cypher (Neo4j) - Run #{RUN_INDEX}

This model is a fine-tuned version of **{BASE_MODEL_NAME}** optimized for translating natural language questions into accurate **Neo4j Cypher queries** using a structured database schema.

Number of samples used: `{end_idx}`

## Model Details

- **Base Architecture:** Qwen2.5-Coder-3B-Instruct
- **Fine-Tuning Method:** QLoRA (Rank 16, Alpha 16)
- **Dataset Used:** `{DATASET_NAME}` ({dataset_size} examples)
- **Epochs:** {TARGET_EPOCHS}
- **Frameworks:** Unsloth, Hugging Face TRL, PEFT, PyTorch

## Prompts & Schema Usage

The model expects system prompts containing the Neo4j schema structure:

```text
<|im_start|>system
You are an expert in Neo4j databases and the Cypher query language. Your task is to convert natural language queries into accurate and efficient Cypher queries based on the provided database schema.

### Database Schema:
[PROVIDE DATABASE SCHEMA HERE]<|im_end|>
<|im_start|>user
[YOUR NATURAL LANGUAGE QUESTION]<|im_end|>
<|im_start|>assistant
```

## Running with Ollama (GGUF)
You can run the exported GGUF model ({HUB_REPO_GGUF}) directly in Ollama:

```bash
ollama run hf.co/{HUB_REPO_GGUF}:Q4_K_M
```
"""
    # Push README to both Merged and GGUF repositories
    for repo_id in [HUB_REPO_FULL, HUB_REPO_GGUF, HUB_REPO_LORA]:
        try:
            card = ModelCard(model_card_text, ignore_metadata_errors=True)
            card.data = card_data
            card.push_to_hub(repo_id, token=hf_token)
            print(f"📝 Model Card successfully updated for: {repo_id}")
        except Exception as e:
            print(f"Failed to update Model Card for {repo_id}: {e}")

    print("\n✅ All operations completed successfully!")

else:
    print(
        f"\nTraining is not yet complete ({current_epoch}/{TARGET_EPOCHS} epochs). "
        "Only the intermediate LoRA version was updated."
    )


Target epochs reached! Exporting 16-bit merged and GGUF models...
1. Uploading merged 16-bit model to: KRadim/Qwen2.5-Coder-3B-text2cypher-Merged


config.json:   0%|          | 0.00/764 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in KRadim/Qwen2.5-Coder-3B-text2cypher-Merged/tokenizer_config.json.
No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub._upload_pipeline|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

... (42 outputs left out)

Repo card metadata block was not found. Setting CardData to empty.
[huggingface_hub.repocard|WARNING]Repo card metadata block was not found. Setting CardData to empty.


📝 Model Card successfully updated for: KRadim/Qwen2.5-Coder-3B-text2cypher-Merged


Repo card metadata block was not found. Setting CardData to empty.
[huggingface_hub.repocard|WARNING]Repo card metadata block was not found. Setting CardData to empty.


📝 Model Card successfully updated for: KRadim/Qwen2.5-Coder-3B-text2cypher-GGUF
📝 Model Card successfully updated for: KRadim/Qwen2.5-Coder-3B-text2cypher-LoRA

✅ All operations completed successfully!
